# Notebook 6 — Phase 2: Agentic AI Contract Risk Analysis

> **Milestone 2** — This notebook demonstrates the full LangGraph + RAG + LLM agentic pipeline.  
> It shows both **Online (OpenRouter)** and **Offline (Ollama)** modes.

---

## Table of Contents
1. [Environment Setup](#1-environment-setup)
2. [Knowledge Base (RAG)](#2-knowledge-base-rag)
3. [LangGraph State Design](#3-langgraph-state-design)
4. [Node 1 — Classification](#4-node-1--classification)
5. [Node 2 — Research (RAG Retrieval)](#5-node-2--research-rag-retrieval)
6. [Node 3 — Reasoning (Online & Offline)](#6-node-3--reasoning-online--offline)
7. [Full Pipeline Run](#7-full-pipeline-run)
8. [Report Output](#8-report-output)
9. [Disclaimer](#9-disclaimer)

---
## 1. Environment Setup

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import json
import os
import sys
from pathlib import Path
from IPython.display import Markdown, display

# ── Make sure project root is on sys.path ─────────────────────────────────────
PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ── Load .env ─────────────────────────────────────────────────────────────────
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

print("✅ Environment loaded")
print(f"   OPENROUTER_API_KEY : {'SET (' + os.environ.get('OPENROUTER_API_KEY', '')[:12] + '…)' if os.environ.get('OPENROUTER_API_KEY') else 'NOT SET'}")
print(f"   OPENROUTER_MODEL   : {os.environ.get('OPENROUTER_MODEL', 'openai/gpt-oss-120b')}")
print(f"   OLLAMA_MODEL       : {os.environ.get('OLLAMA_MODEL', 'qwen3.5:2b')}")
print(f"   OLLAMA_BASE_URL    : {os.environ.get('OLLAMA_BASE_URL', 'http://localhost:11434')}")

In [ ]:
# ── Check Ollama status ───────────────────────────────────────────────────────
from contract_agent.llm.local import check_ollama_health

health = check_ollama_health()
print("Ollama Health Check:")
print(json.dumps(health, indent=2))

---
## 2. Knowledge Base (RAG)

The RAG system uses a curated JSON file of **Legal Best Practices** instead of a heavy vector DB (FAISS/Chroma).  
A `TF-IDF` vectoriser retrieves the **top-3 most relevant** excerpts for each clause.

In [ ]:
# ── Load and preview the knowledge base ──────────────────────────────────────
data_path = PROJECT_ROOT / "data" / "legal_best_practices.json"
with open(data_path, encoding="utf-8") as f:
    kb_records = json.load(f)

print(f"📚 Knowledge Base: {len(kb_records)} records across topics:")
topics = list({r['topic'] for r in kb_records})
for t in sorted(topics):
    print(f"  • {t}")

print("\n--- Sample Record ---")
print(json.dumps(kb_records[0], indent=2))

In [ ]:
# ── Test the retriever ────────────────────────────────────────────────────────
from contract_agent.retrieval.chroma import LegalPracticeRetriever

retriever = LegalPracticeRetriever(top_k=3)

SAMPLE_CLAUSE = """
Either party may terminate this Agreement at any time, 
with or without cause, upon written notice to the other party.
"""

results = retriever.retrieve(SAMPLE_CLAUSE)
print(f"Top-{len(results)} best-practice excerpts retrieved:\n")
for r in results:
    print(f"  [{r['id']} | {r['topic']}] {r['title']}")
    print(f"  Score: {r['score']:.4f}")
    print(f"  Text:  {r['text'][:120]}…\n")

---
## 3. LangGraph State Design

The agent uses a `TypedDict` as shared state that flows through all nodes.

```
AgentState
├── raw_text              : str    — The full contract text
├── confidence_threshold  : float  — Min ML confidence to include a clause
├── mode                  : str    — "online" | "offline"
├── contract_overview     : str    — Auto-generated summary
├── flagged_clauses       : list   — Output of classification node
├── researched            : list   — Flagged + retrieved best practices
├── clause_assessments    : list   — Researched + LLM analysis
├── structured_report     : dict   — Final structured JSON report
├── markdown_report       : str    — Human-readable markdown report
└── error                 : str?   — Error message if any node fails
```

### Graph Architecture

```
START
  │
  ▼
[classify]  ─── TF-IDF + Logistic Regression
  │               Segments clauses, predicts risk & confidence
  ▼
[research]  ─── TF-IDF RAG over legal_best_practices.json
  │               Retrieves top-3 relevant legal guidelines per clause
  ▼
[reason]    ─── LLM (OpenRouter cloud OR Ollama local)
  │               Explains risk, compares to best practice,
  │               proposes safer clause rewrite
  ▼
 END  ───────── Structured JSON + Markdown Report
```

In [ ]:
# ── Show the AgentState schema ────────────────────────────────────────────────
from contract_agent.workflow import AgentState
import typing

print("AgentState fields:")
for field, ftype in AgentState.__annotations__.items():
    print(f"  {field:28s}: {ftype}")

---
## 4. Node 1 — Classification

Uses the trained `best_model.pkl` (Logistic Regression + TF-IDF pipeline) to classify each clause.

In [ ]:
from contract_agent.utils.ml import load_sklearn_pipeline
from contract_agent.utils.text import clean_text, segment_clauses

SAMPLE_CONTRACT = """
CONSULTING SERVICES AGREEMENT

1. Termination
Either party may terminate this Agreement immediately at any time, with or without cause, 
with no notice required. Upon termination, client forfeits all rights to deliverables 
completed up to that point.

2. Indemnification
The Consultant shall not be liable for any damages, losses, or claims arising from the 
services provided. The Client agrees to indemnify and hold harmless the Consultant from 
any and all claims, including attorney's fees.

3. Payment Terms
Invoices are payable within 30 days of receipt. Late payments accrue interest at 2% per month.

4. Confidentiality
All information shared under this agreement is confidential and may not be disclosed 
to third parties without prior written consent.

5. Governing Law
This Agreement shall be governed by the laws of the State of Delaware.
"""

pipeline = load_sklearn_pipeline()
clauses = segment_clauses(SAMPLE_CONTRACT)

print(f"📄 Contract segmented into {len(clauses)} clauses:\n")
for i, clause in enumerate(clauses, 1):
    cleaned = clean_text(clause)
    risk = pipeline.predict([cleaned])[0]
    probs = pipeline.predict_proba([cleaned])[0]
    confidence = max(probs)
    icon = "🔴" if risk == "High" else "🟠" if risk == "Medium" else "🟢"
    print(f"  {icon} Clause {i}: [{risk}] ({confidence:.1%}) — {clause[:80].strip()}…")

---
## 5. Node 2 — Research (RAG Retrieval)

For each flagged clause, the retriever finds the most relevant legal best-practice excerpts.

In [ ]:
# Simulate what the research node does for the first high-risk clause
HIGH_RISK_CLAUSE = """
Either party may terminate this Agreement immediately at any time, with or without cause, 
with no notice required. Upon termination, client forfeits all rights to deliverables 
completed up to that point.
"""

retrieved = retriever.retrieve(HIGH_RISK_CLAUSE)
print("RAG Retrieval Results for Termination Clause:")
print("=" * 60)
for r in retrieved:
    print(f"\n📌 [{r['id']}] {r['title']}")
    print(f"   Topic: {r['topic']}  | Similarity: {r['score']:.4f}")
    print(f"   {r['text']}")

---
## 6. Node 3 — Reasoning (Online & Offline)

The LLM receives:
1. The clause text
2. The ML risk label + confidence
3. The retrieved best-practice excerpts

And returns a structured JSON with `legal_concern`, `comparison_to_best_practice`, `safer_rewrite`.

In [ ]:
# ── Test OFFLINE mode (Ollama) ────────────────────────────────────────────────
from contract_agent.llm.local import analyze_clause_with_ollama

print(f"Testing Ollama ({os.environ.get('OLLAMA_MODEL', 'qwen3.5:2b')})...")
print("This may take 30-60 seconds on first run.\n")

try:
    offline_result = analyze_clause_with_ollama(
        clause_text=HIGH_RISK_CLAUSE,
        risk_label="High",
        confidence=0.89,
        best_practices=retrieved,
    )
    print("✅ Ollama response:")
    print(json.dumps(offline_result, indent=2))
except RuntimeError as e:
    print(f"❌ Ollama not available: {e}")
    print("   Run: ollama serve")

In [ ]:
# ── Test ONLINE mode (OpenRouter) ─────────────────────────────────────────────
from contract_agent.openrouter_client import analyze_clause_with_openrouter

api_key = os.environ.get("OPENROUTER_API_KEY", "")
if not api_key:
    print("⚠️  OPENROUTER_API_KEY not set — skipping online test")
else:
    print(f"Testing OpenRouter ({os.environ.get('OPENROUTER_MODEL', 'openai/gpt-oss-120b')})...")
    try:
        online_result = analyze_clause_with_openrouter(
            clause_text=HIGH_RISK_CLAUSE,
            risk_label="High",
            confidence=0.89,
            best_practices=retrieved,
        )
        print("✅ OpenRouter response:")
        print(json.dumps(online_result, indent=2))
    except Exception as e:
        print(f"❌ OpenRouter error: {e}")

---
## 7. Full Pipeline Run

Run the complete LangGraph graph end-to-end.  
Change `MODE` to `"online"` or `"offline"` to switch backends.

In [ ]:
from contract_agent.workflow import run_hybrid_agent_pipeline

# ── Choose your mode ──────────────────────────────────────────────────────────
MODE = "offline"   # Change to "online" to use OpenRouter

print(f"🚀 Running full LangGraph pipeline in [{MODE.upper()}] mode...\n")

final_state = run_hybrid_agent_pipeline(
    raw_text=SAMPLE_CONTRACT,
    confidence_threshold=0.5,
    mode=MODE,
)

if final_state.get("error"):
    print(f"❌ Pipeline error: {final_state['error']}")
else:
    print("✅ Pipeline complete!")
    breakdown = final_state["structured_report"]["risk_severity_breakdown"]
    print(f"   High:   {breakdown.get('High', 0)} clauses")
    print(f"   Medium: {breakdown.get('Medium', 0)} clauses")
    print(f"   Low:    {breakdown.get('Low', 0)} clauses")
    n = len(final_state["clause_assessments"])
    print(f"   Total assessed: {n} clauses")

---
## 8. Report Output

In [ ]:
# ── Markdown report ───────────────────────────────────────────────────────────
if not final_state.get("error"):
    display(Markdown(final_state["markdown_report"]))

In [ ]:
# ── Structured JSON report ────────────────────────────────────────────────────
if not final_state.get("error"):
    report = final_state["structured_report"]
    # Show without the disclaimer text to keep output concise
    compact = {k: v for k, v in report.items() if k != "disclaimer"}
    print(json.dumps(compact, indent=2)[:3000], "\n[…truncated for display]")

In [ ]:
# ── Save structured report to file ───────────────────────────────────────────
if not final_state.get("error"):
    out_path = PROJECT_ROOT / "data" / "sample_agentic_report.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(final_state["structured_report"], f, indent=2)
    print(f"✅ Report saved to: {out_path}")

---
## 9. Disclaimer

> ⚠️ **Legal Disclaimer**  
> This output is generated by an **academic research prototype** combining classical machine learning risk scores with large-language-model commentary.  
> It is **not legal advice**, not a substitute for a qualified attorney, and may contain errors or omissions.  
> Always consult a licensed legal professional before making decisions based on this analysis.

---
*Phase 2 — Milestone 2 | Intelligent Contract Risk Classification System*